# Experiment Pipeline

Run batch DDSP timbre transfer and WORLD vocoder baseline on the full voice dataset.

1. **DDSP inference** — `run_synthesize_dir` processes all files in `data/raw/voice/FULL/` through the trained DDSP model.
2. **Baseline inference** — `run_vocoder_dir` processes the same dataset using the WORLD vocoder with a source bank from `data/raw/solo_violin/`.

In [ ]:
import logging
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from experiment_pipeline import run_synthesize_dir, run_vocoder_dir

## Paths

Adjust these if your directory layout differs.

In [ ]:
# --- Input ---
INPUT_DIR = PROJECT_ROOT / "data" / "raw" / "voice" / "FULL"
SOURCE_DIR = PROJECT_ROOT / "data" / "raw" / "solo_violin"

# --- DDSP model ---
MODEL_DIR = PROJECT_ROOT / "artifacts" / "solo_instrument"
GIN_FILE = MODEL_DIR / "operative_config-0.gin"

# --- Output ---
DDSP_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "voice" / "Full_transfered"
BASELINE_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "voice" / "Full_baseline"

print(f"Input dir:    {INPUT_DIR}  (exists: {INPUT_DIR.exists()})")
print(f"Source dir:   {SOURCE_DIR}  (exists: {SOURCE_DIR.exists()})")
print(f"Model dir:    {MODEL_DIR}  (exists: {MODEL_DIR.exists()})")
print(f"Gin file:     {GIN_FILE}  (exists: {GIN_FILE.exists()})")
print(f"DDSP output:  {DDSP_OUTPUT_DIR}")
print(f"BL output:    {BASELINE_OUTPUT_DIR}")

## 1. DDSP Timbre Transfer

Loads the model once, then processes all WAV files via feature-level chunking.

In [ ]:
ddsp_result = run_synthesize_dir(
    model_dir=MODEL_DIR,
    gin_file=GIN_FILE,
    input_dir=INPUT_DIR,
    output_dir=DDSP_OUTPUT_DIR,
    auto_adjust=True,
    pitch_shift=0.0,
    loudness_shift=0.0,
)

print(f"\nDDSP — processed: {ddsp_result['processed']}, failed: {ddsp_result['failed']}")
if ddsp_result["failed_files"]:
    print("Failed files:")
    for f in ddsp_result["failed_files"]:
        print(f"  {f}")

## 2. WORLD Vocoder Baseline

Builds a source bank from solo violin recordings, then runs F0 transfer on all target files.

In [ ]:
baseline_result = run_vocoder_dir(
    input_dir=INPUT_DIR,
    output_dir=BASELINE_OUTPUT_DIR,
    source_dir=SOURCE_DIR,
    method="f0",
    seed=42,
)

print(f"\nBaseline — processed: {baseline_result['processed']}, failed: {baseline_result['failed']}")
if baseline_result["failed_files"]:
    print("Failed files:")
    for f in baseline_result["failed_files"]:
        print(f"  {f}")

## Results Summary

In [ ]:
print("=" * 50)
print("Experiment Pipeline Results")
print("=" * 50)
print(f"DDSP:     {ddsp_result['processed']} OK / {ddsp_result['failed']} failed")
print(f"Baseline: {baseline_result['processed']} OK / {baseline_result['failed']} failed")
print(f"\nOutputs saved to:")
print(f"  DDSP:     {DDSP_OUTPUT_DIR}")
print(f"  Baseline: {BASELINE_OUTPUT_DIR}")